In [2]:
from tool_server.utils.utils import *
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tqdm import tqdm
import os
import json
input_data = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/refocus_chartqa_v_bar_wbb_selfbar_reformatted.jsonl"

input_data2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/refocus_chartqa_v_bar_wbb_reformatted.jsonl"

input_data = process_jsonl(input_data)
input_data2 = process_jsonl(input_data2)
# output_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/sft_data/refocus_chartqa_v_bar_wbb_selfbar_candidates.json"
# output_path2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/sft_data/refocus_chartqa_v_bar_wbb_reformatted.json"

refocus_data_dir = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/ReFocus_data/train_chartQA_v_bar_wbb/gpt-4o-2024-08-06-mix_orig_build_train_nogold"


pixel_crop = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/backup/pixelreasonersft_crop_formatted.jsonl"

pixel_groundingdino_crop = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/backup/pixelreasonersft_groundingcrop_formatted.jsonl"

pixel_crop = process_jsonl(pixel_crop)
pixel_groundingdino_crop = process_jsonl(pixel_groundingdino_crop)




In [15]:
def get_statistics(input_data, key="response_messages"):
    sft_data = []
    tool_statistic_dict = {}
    avg_tool_num_list = []
    tool_format_list = []
    for item in tqdm(input_data):
        conversations = item[key]
        
        current_tool_num = 0
        # print(f"Processing conversation: {conversations}")
        if not isinstance(conversations, list):
            conversations = [conversations]
        for conv in conversations:
            # print(f"Processing conversation: {conv}")
            if "content" not in conv or "role" not in conv:
                continue
            contents = conv["content"]
            role = conv["role"]
            if role == "system":
                continue
            for content in contents:
                # print(f"Processing content: {content}")
                text_content = ""
                if isinstance(content, str):
                    text_content = f"{content}"
                elif "text" in content:
                    text_content = f"{content['text']}"
                
                if "<tool_call>" in text_content:
                    tool_json = text_content.split("<tool_call>")[-1].split("</tool_call>")[0]
                    try:
                        tool_content = json.loads(tool_json)
                        tool_name = tool_content["name"]
                        current_tool_num += 1
                        qualify = 1
                    except:
                        tool_name = "invalid"
                        qualify = 0
                    tool_format_list.append(qualify)
                    tool_statistic_dict[tool_name] = tool_statistic_dict.get(tool_name, 0) + 1
        avg_tool_num_list.append(current_tool_num)
    return tool_statistic_dict, avg_tool_num_list, tool_format_list                
               

In [16]:
refocus_selfbar_statistics = get_statistics(input_data, key="response_messages")
refocus_statistics = get_statistics(input_data2, key="response_messages")
pixel_crop_statistics = get_statistics(pixel_crop, key="message_list")
pixel_groundingdino_crop_statistics = get_statistics(pixel_groundingdino_crop, key="message_list")

100%|██████████| 142/142 [00:00<00:00, 43693.87it/s]


In [26]:
tool_statistic_dict, avg_tool_num_list, tool_format_list   = pixel_groundingdino_crop_statistics
avg_tool_num_list = sum(avg_tool_num_list) / len(avg_tool_num_list)
tool_format_list = sum(tool_format_list) / len(tool_format_list)

In [27]:
tool_statistic_dict, avg_tool_num_list, tool_format_list

({'GroundingDINO': 153,
  'Crop': 150,
  'CropImage': 15,
  'OCR': 20,
  'crop': 4,
  'invalid': 2},
 2.408450704225352,
 0.9941860465116279)

In [ ]:
sft_data = []

for item in tqdm(input_data):
    conversations = item["response_messages"]
    origin_qid = item.get("origin_qid", item["qid"].split("pixelreasonersft_")[1])
    qid = item["qid"]
    current_dir = os.path.join(refocus_data_dir, origin_qid)
    
    jpg_files = [f for f in os.listdir(current_dir) 
             if (f.endswith(".jpg") or f.endswith(".png")) and os.path.isfile(os.path.join(current_dir, f))]
    qid_file_name = f"{origin_qid}.jpg"
    jpg_files.sort(key=lambda x: 0 if x == qid_file_name else 1)
    jpg_files = [os.path.join(current_dir, f) for f in jpg_files]
    
    new_conversations = [{"from":"system","value":tool_planning_model_prompt_one_tool_call}]
    image_cnt = 0
    image_indexes = []
    terminate = False
    for conv in conversations:
        new_role = "gpt" if conv["role"] == "assistant" else "human"
        contents = conv["content"]
        new_value = ""
        for content in contents:
            # print(f"Processing content: {content}")
            if isinstance(content, str):
                new_value += content
            elif "image" in content or "image_url" in content:
                new_value += f"\n<image>\n"
                image_cnt += 1
                image_content = "img_1"
                if content.get("image_url"):
                    if isinstance(content["image_url"], str):
                        image_content = content["image_url"]
                    else:
                        image_content = content["image_url"].get("url", "img_1")
                else:
                    image_content = content.get("image", "img_1")
                
                image_index = int(image_content.split("_")[-1])-1
                image_index = 1 if image_index > 1 else image_index
                image_indexes.append(image_index)
            elif "text" in content:
                new_value += f"{content['text']}"
            elif "tool_code" in content or "tool_call" in content:
                terminate = True
                break
            else:
                raise ValueError(f"Unknown content type: {content['type']}")
        if terminate:
            break
        new_conversations.append({"from": new_role, "value": new_value})
    if terminate:
        continue
    
    new_jpg_files = [jpg_files[i] for i in image_indexes]
    assert image_cnt == len(new_jpg_files), f"Image count mismatch for {origin_qid}: {image_cnt} vs {len(jpg_files)}"
    new_item = {
        "qid": qid,
        "conversations": new_conversations,
        "images": new_jpg_files,
    }
    sft_data.append(new_item)
    

100%|██████████| 188/188 [00:00<00:00, 418.48it/s]


In [37]:
write_json_file(sft_data, output_path)
len(sft_data)

185

In [33]:
sft_data = []
for item in tqdm(input_data2):
    conversations = item["response_messages"]
    # origin_qid = item.get("origin_qid", item["qid"].split("refocus_chartqa_v_bar_wbb_candidates/")[1])
    origin_qid = item["origin_qid"]
    qid = item["qid"]
    current_dir = os.path.join(refocus_data_dir, origin_qid)
    
    jpg_files = [f for f in os.listdir(current_dir) 
             if (f.endswith(".jpg") or f.endswith(".png")) and os.path.isfile(os.path.join(current_dir, f))]
    qid_file_name = f"{origin_qid}.jpg"
    jpg_files.sort(key=lambda x: 0 if x == qid_file_name else 1)
    jpg_files = [os.path.join(current_dir, f) for f in jpg_files]
    
    new_conversations = [{"from":"system","value":tool_planning_model_prompt_one_tool_call}]
    image_cnt = 0
    image_indexes = []
    terminate = False
    for conv in conversations:
        new_role = "gpt" if conv["role"] == "assistant" else "human"
        contents = conv["content"]
        new_value = ""
        for content in contents:
            # print(f"Processing content: {content}")
            if isinstance(content, str):
                new_value += content
            elif "image" in content or "image_url" in content:
                new_value += f"\n<image>\n"
                image_cnt += 1
                image_content = "img_1"
                if content.get("image_url"):
                    if isinstance(content["image_url"], str):
                        image_content = content["image_url"]
                    else:
                        image_content = content["image_url"].get("url", "img_1")
                else:
                    image_content = content.get("image", "img_1")
                
                image_index = int(image_content.split("_")[-1])-1
                image_index = 1 if image_index > 1 else image_index
                image_indexes.append(image_index)
            elif "text" in content:
                new_value += f"{content['text']}"
            elif "tool_code" in content or "tool_call" in content or "tool_response" in content or ("type" in content and content["type"] == "tool_call"):
                terminate = True
                break
            else:
                raise ValueError(f"Unknown content type: {content['type']}")
        if terminate:
            break
        new_conversations.append({"from": new_role, "value": new_value})
    if terminate:
        continue
    
    new_jpg_files = [jpg_files[i] for i in image_indexes]
    assert image_cnt == len(new_jpg_files), f"Image count mismatch for {origin_qid}: {image_cnt} vs {len(jpg_files)}"
    new_item = {
        "qid": qid,
        "conversations": new_conversations,
        "images": new_jpg_files,
    }
    sft_data.append(new_item)
    

100%|██████████| 178/178 [00:00<00:00, 405.66it/s]


In [35]:
write_json_file(sft_data, output_path2)
len(sft_data)

150